# 15 Class 2 vs 3: tuned models

Is the model comparison in 14 fair to models that need tuning? Each model gets its settings tuned **inside** every outer training set. Outer loop: the saved folds (first 3 repeats only, to save time). Inner loop: 5 folds that keep twin groups together, scored by AUC. Same 15 shape features, same 160 segments.

Rule: a model beats another only if its mean AUC is more than 0.03 higher and it wins in all 3 repeats. Otherwise it is a tie and the simpler model stays.

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from src.config import PROCESSED, RANDOM_STATE
from src.cv import load_fold_table
from src.evaluate import cross_validate_segments
from src.features import FEATURES, SIZE_FEATURES, SHAPE_FEATURES

feat = pd.read_csv(PROCESSED / "features_dev.csv")
table = load_fold_table("cv_folds_allclass.csv")
data = feat[feat["y"].isin([2, 3])].reset_index(drop=True)          # Kaggle classes 2 and 3, development set only
assert set(data["segment"]) <= set(table["segment"])

models = {
    "chance": lambda: DummyClassifier(strategy="prior"),
    "logistic regression": lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "random forest": lambda: RandomForestClassifier(200, random_state=RANDOM_STATE, n_jobs=-1),
}

import warnings
warnings.filterwarnings("ignore", category=FutureWarning, module="sklearn.svm")
from sklearn.svm import SVC
from sklearn.ensemble import HistGradientBoostingClassifier
from src.evaluate import cross_validate_segments_tuned

N_REP = 3
family = {
    "logistic regression": (lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
                            {"logisticregression__C": [0.01, 0.1, 1, 10]}),
    "SVM (RBF)": (lambda: make_pipeline(StandardScaler(), SVC(probability=True, random_state=RANDOM_STATE)),
                  {"svc__C": [0.1, 1, 10], "svc__gamma": [0.01, 0.03, 0.1, 0.3]}),
    "random forest": (lambda: RandomForestClassifier(200, random_state=RANDOM_STATE, n_jobs=1),
                      {"max_depth": [3, 5, None], "min_samples_leaf": [1, 3, 5]}),
    "gradient boosting": (lambda: HistGradientBoostingClassifier(max_iter=100, learning_rate=0.05, max_depth=3, random_state=RANDOM_STATE),
                          {}),       # left at defaults
}
default, tuned, chosen = {}, {}, {}
for name, (mk, grid) in family.items():
    default[name], _ = cross_validate_segments_tuned(mk, {}, data, SHAPE_FEATURES, table, n_repeats=N_REP, positive=2)
    tuned[name], chosen[name] = cross_validate_segments_tuned(mk, grid, data, SHAPE_FEATURES, table, n_repeats=N_REP, positive=2)
    print(name, "default", round(default[name]["auc_fold"].mean(), 3), "| tuned", round(tuned[name]["auc_fold"].mean(), 3))

logistic regression default 0.758 | tuned 0.767


SVM (RBF) default 0.788 | tuned 0.777


random forest default 0.789 | tuned 0.761


gradient boosting default 0.711 | tuned 0.711


## Tuned vs default (same 3 repeats)

In [2]:
d_auc = pd.DataFrame({n: r["auc_fold"] for n, r in default.items()})
t_auc = pd.DataFrame({n: r["auc_fold"] for n, r in tuned.items()})
gain = t_auc - d_auc
display(pd.DataFrame({"default AUC": d_auc.mean(), "tuned AUC": t_auc.mean(), "tuned - default": gain.mean(),
                      "repeats tuned better (of 3)": (gain > 0).sum()}).round(3))

,default AUC,tuned AUC,tuned - default,repeats tuned better (of 3)
logistic regression,0.758,0.767,0.008,2
SVM (RBF),0.788,0.777,-0.011,1
random forest,0.789,0.761,-0.028,0
gradient boosting,0.711,0.711,0.000,0


## Tuned models vs tuned logistic

In [3]:
diff = t_auc.sub(t_auc["logistic regression"], axis=0)
cmp = pd.DataFrame({"tuned AUC": t_auc.mean(), "sd over repeats": t_auc.std(), "diff vs tuned logistic": diff.mean(),
                    "repeats better (of 3)": (diff > 0).sum()})
cmp["beats logistic by the rule"] = (cmp["diff vs tuned logistic"] > 0.03) & (cmp["repeats better (of 3)"] == 3)
cmp.loc["logistic regression", "beats logistic by the rule"] = False
display(cmp.round(3))
display(pd.DataFrame({n: r[["pr_auc_fold", "sensitivity", "specificity", "balanced_accuracy"]].mean() for n, r in tuned.items()}).T.round(3))

,tuned AUC,sd over repeats,diff vs tuned logistic,repeats better (of 3),beats logistic by the rule
logistic regression,0.767,0.044,0.000,0,False
SVM (RBF),0.777,0.003,0.010,2,False
random forest,0.761,0.044,-0.006,1,False
gradient boosting,0.711,0.029,-0.055,0,False


,pr_auc_fold,sensitivity,specificity,balanced_accuracy
logistic regression,0.824,0.654,0.721,0.688
SVM (RBF),0.819,0.704,0.746,0.725
random forest,0.799,0.700,0.729,0.715
gradient boosting,0.760,0.696,0.662,0.679


## Which settings were chosen (30 outer folds each)

In [4]:
for name, ch in chosen.items():
    if ch.empty:
        continue
    cols = [c for c in ch.columns if c not in ("repeat", "fold")]
    print(name)
    display(ch.fillna("None").astype(str).groupby(cols).size().rename("times chosen").reset_index().sort_values("times chosen", ascending=False))

logistic regression


,logisticregression__C,times chosen
2,10.0,26
0,0.1,3
1,1.0,1


SVM (RBF)


,svc__C,svc__gamma,times chosen
5,10.0,0.03,15
4,10.0,0.01,7
0,0.1,0.1,3
3,1.0,0.3,2
1,0.1,0.3,1
2,1.0,0.1,1
6,10.0,0.1,1


random forest


,max_depth,min_samples_leaf,times chosen
5,None,1,18
3,5.0,3,4
0,3.0,1,2
2,5.0,1,2
1,3.0,5,1
4,5.0,5,1
6,None,3,1
7,None,5,1


## Findings

| Model | default AUC | tuned AUC | diff vs tuned logistic |
|---|---|---|---|
| logistic regression | 0.758 | 0.767 | 0 |
| SVM (RBF) | 0.788 | 0.777 | +0.010 |
| random forest | 0.789 | 0.761 | -0.006 |
| gradient boosting (not tuned) | 0.711 | 0.711 | -0.055 |

- Tuning does not help. Changes from default are within about 0.03, and for SVM and forest the tuned score is slightly lower. Choosing settings on small inner folds adds noise.
- No tuned model beats tuned logistic by the rule. SVM is +0.010 and wins 2 of 3 repeats, so it is a tie. Same answer as the fresh-fold check in 14. **Logistic regression stays as the model.**
- Settings picked: logistic nearly always took the weakest regularisation, C = 10 (26 of 30), the edge of the grid. The SVM mostly took C = 10 as well. The forest mostly took full-depth trees with 1 sample per leaf (18 of 30). Little regularisation is preferred, which is what you would expect with only 15 features.

## Caveats

- 3 repeats only, so "wins in all 3" is a weak condition. The sd over repeats (0.044 for logistic) is larger than most gaps.
- The grids are small and were fixed in advance. A logistic C grid that stops at 10 may have been too short.
- SVC(probability=True) runs its own internal 5-fold calibration, not twin-aware. It only touches the training data, so it cannot leak into the validation fold, but the calibration may be slightly off.
- Same patient-overlap limit as before. Scores are upper bounds.